# BirdCLEF 2026 — Distilled SED (Colab Pro+ 版)

**変更点 (vs Kaggle版):**
- Perch 埋め込みを学習前に一括プリ計算 → 学習ループ中の ONNX 推論を排除
- 学習が GPU バウンドになり 5〜10x 高速化
- Google Drive にデータ・出力を永続化

**初回セットアップ手順:**
1. Google Drive の `MyDrive/birdclef2026/` に `kaggle.json` をアップロード
2. 「ランタイム → すべてのセルを実行」

In [ ]:
# =================================================================
# COLAB SETUP — Secrets 認証のみ
# =================================================================
import sys, os

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_COLAB:
    from google.colab import userdata

    DATA_DIR = '/content/data'
    OUT_DIR  = '/content/working'
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(OUT_DIR,  exist_ok=True)

    os.system('pip install -q kaggle')
    os.makedirs('/root/.kaggle', exist_ok=True)
    kaggle_key = userdata.get('KAGGLE_KEY')
    kaggle_json = f'{{"username":"gorubachohu","key":"{kaggle_key}"}}'
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(kaggle_json)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle auth: OK')
    print(f'DATA_DIR : {DATA_DIR}')
    print(f'OUT_DIR  : {OUT_DIR}')

In [ ]:
# =================================================================
# DL-1: birdclef-2026 コンペデータ（~5GB、unzip あり）
# =================================================================
import os, subprocess, sys

DATA_DIR = '/content/data'
dst = f'{DATA_DIR}/birdclef-2026'
if os.path.exists(dst):
    print(f'already exists: {dst}')
else:
    print('Downloading birdclef-2026 ...')
    subprocess.run(
        f'kaggle competitions download -c birdclef-2026 -p {DATA_DIR}/',
        shell=True)
    print('Unzipping ...')
    subprocess.run(
        f'unzip -q {DATA_DIR}/birdclef-2026.zip -d {dst}/ && rm -f {DATA_DIR}/birdclef-2026.zip',
        shell=True)
    print(f'Done: {dst}')
    subprocess.run(f'du -sh {dst}', shell=True)

In [ ]:
# =================================================================
# DL-2: waveform cache（大容量）— zip のまま保持、unzip しない
#   → DL-3 セルで CSV だけ展開して .pt は zip から直接読む
# =================================================================
import os, subprocess, time

DATA_DIR = '/content/data'
ZIP_PATH = f'{DATA_DIR}/birdclef-2026-waveform-cache.zip'

if os.path.exists(ZIP_PATH):
    size = os.path.getsize(ZIP_PATH) / 1e9
    print(f'already exists: {ZIP_PATH} ({size:.1f} GB)')
else:
    print('Downloading waveform cache (zip のまま) ...')
    t0 = time.time()
    subprocess.run(
        f'kaggle datasets download -d tuckerarrants/birdclef-2026-waveform-cache -p {DATA_DIR}/',
        shell=True)
    elapsed = time.time() - t0
    if os.path.exists(ZIP_PATH):
        size = os.path.getsize(ZIP_PATH) / 1e9
        print(f'Done: {ZIP_PATH} ({size:.1f} GB, {elapsed/60:.1f} min)')
    else:
        print('ERROR: zip not found after download')

In [ ]:
# =================================================================
# DL-3: waveform cache の CSV だけ展開（.pt は zip から直接読む）
# =================================================================
import os, zipfile

DATA_DIR = '/content/data'
ZIP_PATH = f'{DATA_DIR}/birdclef-2026-waveform-cache.zip'
CSV_DIR  = f'{DATA_DIR}/waveform_cache_csv'
os.makedirs(CSV_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    all_files = zf.namelist()
    csv_files = [f for f in all_files if f.endswith('.csv')]
    pt_count  = sum(1 for f in all_files if f.endswith('.pt'))
    print(f'zip contents: {len(all_files)} files total, {pt_count} .pt files, {len(csv_files)} CSV files')
    print('CSV files:', csv_files)
    for csv in csv_files:
        zf.extract(csv, CSV_DIR)
        extracted = os.path.join(CSV_DIR, csv)
        print(f'  extracted: {extracted}')

# zip 内の .pt ファイルのパス prefix を確認
sample_pts = [f for f in all_files if f.endswith('.pt')][:3]
print('Sample .pt paths in zip:', sample_pts)

In [ ]:
# =================================================================
# DL-4: perch ONNX + B1 weights（小さいのでunzipあり）
# =================================================================
import os, subprocess

DATA_DIR = '/content/data'

def _dl_small(slug, kind, unzip_to):
    base     = slug.split('/')[-1]
    zip_path = f'{DATA_DIR}/{base}.zip'
    if os.path.exists(unzip_to):
        print(f'already exists: {unzip_to}')
        return
    print(f'Downloading {slug} ...')
    cmd = f'kaggle competitions download -c {slug} -p {DATA_DIR}/' if kind == 'competition' \
          else f'kaggle datasets download -d {slug} -p {DATA_DIR}/'
    subprocess.run(cmd, shell=True)
    subprocess.run(f'unzip -q {zip_path} -d {unzip_to}/ && rm -f {zip_path}', shell=True)
    print(f'Done: {unzip_to}')

_dl_small('tuckerarrants/perch-v2-no-dft-onnx',           'dataset', f'{DATA_DIR}/perch-onnx')
_dl_small('gorubachohu/timm-efficientnet-b1-ns-jft-in1k', 'dataset', f'{DATA_DIR}/b1-weights')
print('All downloads complete!')

In [ ]:
# packages
if IS_COLAB:
    import subprocess, os
    # /tmp のキャッシュを使わない（tmpfs が小さいため）
    subprocess.run('pip install -q --no-cache-dir onnxruntime-gpu timm torchaudio onnx safetensors', shell=True)
    # torchinductor キャッシュを /content に移動
    os.environ['TORCHINDUCTOR_CACHE_DIR'] = '/content/torchinductor_cache'
    os.makedirs('/content/torchinductor_cache', exist_ok=True)
else:
    # Kaggle: onnxruntime wheel is bundled
    pass

## S1 — Imports & Config

In [ ]:
# =================================================================
# S1 -- IMPORTS + CONFIG
# =================================================================
import os, sys, time, json, pickle, gc, random, math, zipfile, io as _io
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
from scipy.special import expit as sigmoid_np
import warnings
warnings.filterwarnings('ignore')

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    cc = torch.cuda.get_device_capability(0)
    print(f'CC:      sm_{cc[0]}{cc[1]}')

device = torch.device('cuda')
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
if IS_COLAB:
    DATA_DIR             = '/content/data'
    COMP_DIR             = Path(f'{DATA_DIR}/birdclef-2026')
    # waveform cache: .pt は zip から読む、CSV は展開済み
    WAVEFORM_CACHE_ZIP   = f'{DATA_DIR}/birdclef-2026-waveform-cache.zip'
    WAVEFORM_CACHE_DIR   = Path(f'{DATA_DIR}/waveform_cache/waveform_cache')  # 展開済みの場合はこちら
    WAVEFORM_CSV_DIR     = Path(f'{DATA_DIR}/waveform_cache_csv')
    PERCH_ONNX_PATH      = Path(f'{DATA_DIR}/perch-onnx/perch_v2_no_dft.onnx')
    B1_WEIGHTS_PATH      = f'{DATA_DIR}/b1-weights/model.safetensors'
    OUT_DIR              = Path('/content/working')
else:  # Kaggle
    COMP_DIR             = Path('/kaggle/input/competitions/birdclef-2026')
    WAVEFORM_CACHE_ZIP   = None
    WAVEFORM_CACHE_DIR   = Path('/kaggle/input/datasets/tuckerarrants/birdclef-2026-waveform-cache/waveform_cache')
    WAVEFORM_CSV_DIR     = WAVEFORM_CACHE_DIR
    PERCH_ONNX_PATH      = Path('/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx')
    B1_WEIGHTS_PATH      = '/kaggle/input/timm-efficientnet-b1-ns-jft-in1k/model.safetensors'
    OUT_DIR              = Path('/kaggle/working')

OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_PATH     = COMP_DIR / 'train_soundscapes_labels.csv'
TAXONOMY_PATH   = COMP_DIR / 'taxonomy.csv'
SAMPLE_SUB_PATH = COMP_DIR / 'sample_submission.csv'
TEST_DIR        = COMP_DIR / 'test_soundscapes'

NUM_CLASSES = 234
SR          = 32000

TRAIN_DURATION = 5
VAL_DURATION   = 5
TRAIN_SAMPLES  = SR * TRAIN_DURATION  # 160000
VAL_SAMPLES    = SR * VAL_DURATION

N_FOLDS    = 5
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 256
FMIN       = 20
FMAX       = 16000

BACKBONE_NAME     = 'tf_efficientnet_b1.ns_jft_in1k'
USE_PERCH_DISTILL = True
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0

MODE   = 'train'
DEBUG  = False
FOLDS  = [0]
EPOCHS = 6
BATCH  = 32
LR     = 5e-4
MIN_LR = 1e-5
WD     = 1e-4
WARMUP_EPOCHS = 2

MIN_SAMPLE = 20
AUG_PROB   = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

USE_FOCAL_MIXUP        = True
MIXUP_PROB             = 0.5
MIXUP_ALPHA            = 0.4
MIXUP_HARD             = True
USE_FOCAL_SC_MIXUP     = True
FOCAL_SC_MIXUP_PROB    = 0.5
FOCAL_SC_MIXUP_ALPHA   = 0.4

FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

USE_FOCAL           = True
USE_FOCAL_SECONDARY = True
USE_LABELED_SC      = True
SHARES = {'focal': 0.9, 'sc': 0.1}
SOURCE_WEIGHTS = {'focal': 1.0, 'focal_missing': 0.0, 'sc': 1.0}

EMBED_CACHE = {}

# ------------------------------------------------------------------
# zip 内 .pt パスの prefix を自動検出
# ------------------------------------------------------------------
_ZIP_PREFIX = ''  # 例: 'waveform_cache/' or ''
if IS_COLAB and WAVEFORM_CACHE_ZIP and os.path.exists(WAVEFORM_CACHE_ZIP):
    with zipfile.ZipFile(WAVEFORM_CACHE_ZIP, 'r') as _zf:
        _pts = [f for f in _zf.namelist() if f.endswith('.pt')]
        if _pts:
            _ZIP_PREFIX = _pts[0].rsplit('/', 1)[0] + '/' if '/' in _pts[0] else ''
    print(f'zip .pt prefix: "{_ZIP_PREFIX}"  ({len(_pts)} .pt files)')

# zip から .pt を読むグローバルハンドル（main process 用）
_ZF_HANDLE = None
def _get_zf():
    global _ZF_HANDLE
    if _ZF_HANDLE is None and WAVEFORM_CACHE_ZIP and os.path.exists(WAVEFORM_CACHE_ZIP):
        _ZF_HANDLE = zipfile.ZipFile(WAVEFORM_CACHE_ZIP, 'r')
    return _ZF_HANDLE

def load_int16_from_zip(relative_path):
    """zip から .pt を読む。展開済みファイルがあればそちらを優先。"""
    disk_path = WAVEFORM_CACHE_DIR / relative_path
    if disk_path.exists():
        return torch.load(disk_path, map_location='cpu').float() / 32767.0
    zf = _get_zf()
    if zf is None:
        return None
    try:
        data = zf.read(f'{_ZIP_PREFIX}{relative_path}')
        return torch.load(_io.BytesIO(data), map_location='cpu').float() / 32767.0
    except Exception:
        return None

print(f'Backbone : {BACKBONE_NAME}')
print(f'Epochs   : {EPOCHS}  Batch: {BATCH}  Folds: {FOLDS}')
print(f'OUT_DIR  : {OUT_DIR}')

## S2 — Load Data

In [ ]:
# =================================================================
# S2 -- LOAD DATA
# =================================================================
import onnxruntime as ort

sample_sub    = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX     = {l: i for i, l in enumerate(PRIMARY_LABELS)}
taxonomy      = pd.read_csv(TAXONOMY_PATH)
label_to_taxon = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name'].astype(str)))
TAXON_MASKS   = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS) if label_to_taxon.get(l,'') == t])
                 for t in ['Aves','Amphibia','Insecta','Mammalia','Reptilia']}

# CSV は WAVEFORM_CSV_DIR から読む（zip から展開済み or Kaggle の場合は WAVEFORM_CACHE_DIR）
def _csv(name):
    p = WAVEFORM_CSV_DIR / name
    if p.exists(): return p
    return WAVEFORM_CACHE_DIR / name  # Kaggle fallback

audio_cache_meta = pd.read_csv(_csv('audio_cache_meta.csv'))
train_df         = pd.read_csv(COMP_DIR / 'train.csv')
audio_cache_meta = audio_cache_meta.merge(train_df[['filename','secondary_labels']], on='filename', how='left')
audio_cache_meta = audio_cache_meta[audio_cache_meta['primary_label'].isin(LABEL2IDX)].reset_index(drop=True)
print(f'Focal cache: {len(audio_cache_meta)} entries')

sc_cache_meta = pd.read_csv(_csv('soundscape_cache_meta.csv'))
sc_cache_meta['label_list'] = sc_cache_meta['label_list'].apply(
    lambda x: x.split(';') if isinstance(x, str) else [])
print(f'SC cache: {len(sc_cache_meta)} windows')

sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
sc_labels_raw['start_sec'] = pd.to_timedelta(sc_labels_raw['start']).dt.total_seconds().astype(int)
Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    matches = sc_labels_raw[(sc_labels_raw['filename'] == row['filename']) &
                             (sc_labels_raw['start_sec'] == row['start_sec'])]
    for _, m in matches.iterrows():
        for lbl in str(m['primary_label']).split(';'):
            lbl = lbl.strip()
            if lbl in LABEL2IDX: Y_SC[i, LABEL2IDX[lbl]] = 1.0
labeled_sc_mask = Y_SC.sum(axis=1) > 0
print(f'SC labels: {labeled_sc_mask.sum()}/{len(Y_SC)} labeled')

audio_for_split = audio_cache_meta.drop_duplicates('original_idx').reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split['fold'] = -1
for fold, (_, vi) in enumerate(skf.split(audio_for_split, audio_for_split['primary_label'])):
    audio_for_split.loc[vi, 'fold'] = fold
audio_cache_meta = audio_cache_meta.merge(audio_for_split[['original_idx','fold']], on='original_idx', how='left')

sc_files = sc_cache_meta[['filename','site']].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files['fold'] = -1
for fold, (_, vi) in enumerate(gkf.split(sc_files, groups=sc_files['filename'])):
    sc_files.loc[sc_files.index[vi], 'fold'] = fold
file_to_fold = dict(zip(sc_files['filename'], sc_files['fold']))
sc_cache_meta['fold'] = sc_cache_meta['filename'].map(file_to_fold).fillna(-1).astype(int)

counts = audio_cache_meta['primary_label'].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index
extra = []
for sp in rare_species:
    rows = audio_cache_meta[audio_cache_meta['primary_label'] == sp]
    for _ in range(int(np.ceil(MIN_SAMPLE / len(rows))) - 1):
        extra.append(rows)
n_before = len(audio_cache_meta)
if extra:
    audio_cache_meta = pd.concat([audio_cache_meta] + extra, ignore_index=True)
print(f'Upsampled {len(rare_species)} rare species: {n_before} -> {len(audio_cache_meta)}')

non_s22_mask_sc = sc_cache_meta['site'].values != 'S22'
print(f'S22 mask: non-S22={non_s22_mask_sc.sum()}')
print('OK Data loaded')

In [ ]:
if DEBUG:
    EPOCHS = 1; FOLDS = [0]
    audio_cache_meta = audio_cache_meta.groupby('primary_label').head(3).reset_index(drop=True)
    sc_cache_meta = sc_cache_meta.head(50)
    Y_SC = Y_SC[:50]; non_s22_mask_sc = non_s22_mask_sc[:50]
    print(f'DEBUG: {len(audio_cache_meta)} focal, {len(sc_cache_meta)} sc')

## S3 — Model Architecture

In [ ]:
# =================================================================
# S3 -- EVAL UTILITIES + MEL + SED MODEL
# =================================================================

def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    if mask is not None: y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None: y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col): continue
        try: aucs.append(roc_auc_score(col, y_pred[:, c]))
        except: continue
    return (np.mean(aucs) if aucs else float('nan')), len(aucs)

def full_eval(y_true, y_pred, ns22, tm):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r['macro_auc_all'], r['n_all'] = round(a, 4), n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22)
    r['non_s22_macro'], r['n_ns22'] = round(a, 4), n
    for t, cm in tm.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22, class_mask=cm)
        r[f'non_s22_{t}'] = round(a, 4)
    return r

class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x):
        return self.db_transform(self.mel_spec(x))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)
    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS): mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS): mel = self.time_mask(mel)
        return mel

class PerchTeacher:
    def __init__(self, onnx_path):
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._embed_idx = next(
            (i for i, o in enumerate(self.session.get_outputs()) if o.shape and o.shape[-1] == PERCH_EMBED_DIM),
            1)
        print(f'Perch loaded: embed_idx={self._embed_idx}, providers={self.session.get_providers()}')
    @torch.no_grad()
    def embed(self, waveforms_5s):
        wav_np = waveforms_5s.cpu().numpy() if isinstance(waveforms_5s, torch.Tensor) else waveforms_5s
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))

def load_backbone_weights_offline(backbone):
    if not os.path.exists(B1_WEIGHTS_PATH):
        print(f'  WARNING: weights not found at {B1_WEIGHTS_PATH}')
        return
    from safetensors.torch import load_file as _st_load
    sd = _st_load(B1_WEIGHTS_PATH)
    for k in list(sd.keys()):
        if sd[k].ndim == 4 and sd[k].shape[1] == 3:
            bk = dict(backbone.named_parameters()).get(k)
            if bk is not None and bk.shape[1] == 1:
                sd[k] = sd[k].mean(dim=1, keepdim=True)
                print(f'  Adapted {k}: 3ch->1ch')
    missing, unexpected = backbone.load_state_dict(sd, strict=False)
    print(f'  Loaded B1 weights: {len(missing)} missing, {len(unexpected)} unexpected')

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)

class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool='', drop_path_rate=drop_path_rate)
        load_backbone_weights_offline(self.backbone)
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            feat = self.backbone(torch.randn(1, 1, N_MELS, n_tf))
            self.backbone_dim = feat.shape[1]
            print(f'Backbone feat: {tuple(feat.shape)}')
        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25), nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.att = nn.Conv1d(hidden_dim, num_classes, 1)
        self.cla = nn.Conv1d(hidden_dim, num_classes, 1)
        nn.init.xavier_uniform_(self.att.weight); self.att.bias.data.fill_(0.)
        nn.init.xavier_uniform_(self.cla.weight); self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = self.distill_head(h) if (return_distill and hasattr(self, 'distill_head')) else None
        h_cls = h.detach() if USE_PERCH_DISTILL else h
        h_cls = self.gem_freq(h_cls).permute(0, 2, 1)
        h_cls = self.dense(h_cls).permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill: return clip_logits, fw, distill_emb
        elif return_framewise: return clip_logits, fw
        elif return_distill:   return clip_logits, distill_emb
        return clip_logits

def make_model():
    return BirdSEDModel(BACKBONE_NAME).to(device)

print('OK Model definitions ready')

## S4 — Data Pipeline (embed_cache 対応)

In [ ]:
# =================================================================
# S4 -- DATA PIPELINE (EMBED_CACHE 対応 + zip 読み込み)
# =================================================================

def extract_chunk_np(waveform, start_sample, n_samples):
    total = len(waveform)
    if total <= n_samples: return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total: start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w

# --- ファイル読み込み（disk 優先、なければ zip から）---
_FC = {}
def load_focal(p):
    if p in _FC: return _FC[p]
    a = load_int16_from_zip(p)
    if a is None: return None
    if isinstance(a, torch.Tensor): a = a.numpy()
    if len(_FC) >= 3000: _FC.pop(next(iter(_FC)))
    _FC[p] = a
    return a

_SC_CACHE = {}
def load_sc_waveform(cache_file):
    if cache_file in _SC_CACHE: return _SC_CACHE[cache_file]
    a = load_int16_from_zip(cache_file)
    if a is None: return None
    if isinstance(a, torch.Tensor): a = a.numpy()
    if len(_SC_CACHE) >= 300: _SC_CACHE.pop(next(iter(_SC_CACHE)))
    _SC_CACHE[cache_file] = a
    return a

def _focal_embed(cache_file):
    key = f'focal/{cache_file}'
    emb = EMBED_CACHE.get(key)
    return torch.from_numpy(emb.copy()) if emb is not None else torch.zeros(PERCH_EMBED_DIM)

def _sc_embed(filename, start_sec):
    key = f'sc/{filename}/{start_sec}'
    emb = EMBED_CACHE.get(key)
    return torch.from_numpy(emb.copy()) if emb is not None else torch.zeros(PERCH_EMBED_DIM)

# SC MixUp pool
sc_mixup_sources = []
_sc_file_meta = pd.read_csv(_csv('soundscape_file_meta.csv'))
_sc_file_dict  = dict(zip(_sc_file_meta['filename'], _sc_file_meta['cache_file']))
_labeled_rows  = []
for i in range(len(sc_cache_meta)):
    row = sc_cache_meta.iloc[i]
    if Y_SC[i].sum() > 0:
        cf = _sc_file_dict.get(row['filename'])
        if cf is not None:
            _labeled_rows.append({'filename': row['filename'], 'start_sec': int(row['start_sec']),
                                   'cache_file': cf, 'label_idx': i, 'fold': int(row.get('fold', -1))})
if _labeled_rows:
    _labeled_meta = pd.DataFrame(_labeled_rows)
    sc_mixup_sources.append((_labeled_meta, Y_SC))
    print(f'SC MixUp pool: {len(_labeled_meta)} windows')

class FocalDS(Dataset):
    def __init__(self, df, l2i, secondary_lookup=None, sc_mixup_sources=None, fold_k=None, aug=False):
        self.df   = df.reset_index(drop=True)
        self.l2i  = l2i
        self.aug  = aug
        self.secondary_lookup  = secondary_lookup
        self.sc_mixup_sources  = sc_mixup_sources
        self.fold_k            = fold_k

    def __len__(self): return len(self.df)

    def _load_chunk(self, r):
        w = load_focal(r['cache_file'])
        if w is None: return None, None
        start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if self.aug and len(w) > TRAIN_SAMPLES else 0
        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r['primary_label']) in self.l2i:
            lb[self.l2i[str(r['primary_label'])]] = 1.0
        if self.secondary_lookup is not None and 'original_idx' in self.df.columns:
            for s in self.secondary_lookup.get(int(r['original_idx']), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return ch, lb

    def __getitem__(self, i):
        r1  = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r1)
        emb1 = _focal_embed(r1['cache_file'])

        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES),
                    'focal_missing', torch.zeros(PERCH_EMBED_DIM))

        if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2 = None
            for _ in range(3):
                j   = np.random.randint(len(self.df))
                r2  = self.df.iloc[j]
                ch2, lb2 = self._load_chunk(r2)
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb  = np.maximum(lb1, lb2) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb2
                emb2 = _focal_embed(r2['cache_file'])
                emb_mix = lam * emb1 + (1 - lam) * emb2
                return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'focal', emb_mix)

        if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
                and np.random.random() < FOCAL_SC_MIXUP_PROB):
            meta_df_sc, labels = self.sc_mixup_sources[np.random.randint(len(self.sc_mixup_sources))]
            eligible = meta_df_sc[meta_df_sc['fold'] != self.fold_k] if self.fold_k is not None else meta_df_sc
            if len(eligible) > 0:
                sc_row = eligible.iloc[np.random.randint(len(eligible))]
                sc_wav = load_sc_waveform(sc_row['cache_file'])
                if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
                    sc_chunk = extract_chunk_np(sc_wav, int(sc_row['start_sec']) * SR, TRAIN_SAMPLES)
                    lam = np.random.beta(FOCAL_SC_MIXUP_ALPHA, FOCAL_SC_MIXUP_ALPHA)
                    ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
                    if self.aug: ch_mix = apply_aug(ch_mix)
                    lb_sc = labels[int(sc_row['label_idx'])].astype(np.float32)
                    lb    = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
                    emb_sc  = _sc_embed(sc_row['filename'], int(sc_row['start_sec']))
                    emb_mix = lam * emb1 + (1 - lam) * emb_sc
                    return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                            torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'focal', emb_mix)

        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0), torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'focal', emb1)


class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y   = Y
        self.df  = sc_df.reset_index(drop=True)
        self.aug = aug

    def __len__(self): return len(self.Y)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        cf  = _sc_file_dict.get(row['filename'])
        wav_full = load_sc_waveform(cf) if cf else None
        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row['start_sec']) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
        emb = _sc_embed(row['filename'], int(row['start_sec']))
        return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'sc', emb)


class MixSamp(torch.utils.data.Sampler):
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs: per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]: self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                batch.extend([off + int(i) for i in self.rng.integers(0, size, size=n)])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch],
            torch.stack([b[5] for b in batch]))

def mk_sw(sr):
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

focal_secondary_labels = None
if USE_FOCAL_SECONDARY:
    focal_secondary_labels = {}
    for idx, row in train_df.iterrows():
        sec = row.get('secondary_labels', '')
        if pd.isna(sec) or sec in ('', '[]'): continue
        try: sec_list = eval(sec) if isinstance(sec, str) else []
        except: continue
        valid = [s for s in sec_list if s in LABEL2IDX]
        if valid: focal_secondary_labels[idx] = valid
    print(f'Secondary labels: {len(focal_secondary_labels)} files')

print('OK Data pipeline ready')

## S5 — Perch プリ計算 + 学習

In [ ]:
# =================================================================
# S5 -- PERCH PRECOMPUTE + TRAINING
# =================================================================

def precompute_all_embeddings(perch_teacher):
    """全訓練サンプルの Perch 埋め込みを一括計算して EMBED_CACHE に格納"""
    global EMBED_CACHE
    EMBED_CACHE = {}
    EMBED_BATCH = 64
    t0 = time.time()

    # --- Focal: unique original_idx ごとに 1 埋め込み ---
    unique_focal = audio_cache_meta.drop_duplicates('original_idx').reset_index(drop=True)
    print(f'[Precompute] focal: {len(unique_focal)} files (batch={EMBED_BATCH})')
    batch_wavs, batch_keys = [], []

    for i, (_, row) in enumerate(unique_focal.iterrows()):
        wav = load_focal(row['cache_file'])
        if wav is None: continue
        chunk = extract_chunk_np(wav, 0, TRAIN_SAMPLES).astype(np.float32)
        if len(chunk) < 160000: chunk = np.pad(chunk, (0, 160000 - len(chunk)))
        batch_wavs.append(chunk)
        batch_keys.append(f"focal/{row['cache_file']}")

        if len(batch_wavs) >= EMBED_BATCH:
            embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
            for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e
            batch_wavs, batch_keys = [], []

        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            eta = (len(unique_focal) - i - 1) / ((i + 1) / elapsed)
            print(f'  focal {i+1}/{len(unique_focal)} | {elapsed:.0f}s elapsed | ETA {eta/60:.1f}min')

    if batch_wavs:
        embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
        for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e

    # --- SC windows ---
    sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / 'soundscape_file_meta.csv')
    sc_fd = dict(zip(sc_file_meta['filename'], sc_file_meta['cache_file']))
    print(f'[Precompute] SC: {len(sc_cache_meta)} windows')
    batch_wavs, batch_keys = [], []

    for i, row in sc_cache_meta.iterrows():
        cf = sc_fd.get(row['filename'])
        if cf is None: continue
        wav = load_sc_waveform_from(WAVEFORM_CACHE_DIR, cf)
        if wav is None: continue
        chunk = extract_chunk_np(wav, int(row['start_sec']) * SR, TRAIN_SAMPLES).astype(np.float32)
        if len(chunk) < 160000: chunk = np.pad(chunk, (0, 160000 - len(chunk)))
        batch_wavs.append(chunk)
        batch_keys.append(f"sc/{row['filename']}/{int(row['start_sec'])}")

        if len(batch_wavs) >= EMBED_BATCH:
            embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
            for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e
            batch_wavs, batch_keys = [], []

    if batch_wavs:
        embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
        for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e

    elapsed = time.time() - t0
    print(f'[Precompute] Done: {len(EMBED_CACHE)} embeddings in {elapsed:.0f}s ({elapsed/60:.1f}min)')


def _load_val_waveforms(val_sc_df):
    sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / 'soundscape_file_meta.csv')
    sc_file_dict = dict(zip(sc_file_meta['filename'], sc_file_meta['cache_file']))
    wavs = []
    for _, row in val_sc_df.iterrows():
        cf = sc_file_dict.get(row['filename'])
        if cf:
            w = load_sc_waveform_from(WAVEFORM_CACHE_DIR, cf)
            if w is not None:
                wavs.append(torch.from_numpy(extract_chunk_np(w, int(row['start_sec']) * SR, VAL_SAMPLES).astype(np.float32)).unsqueeze(0))
            else: wavs.append(torch.zeros(1, VAL_SAMPLES))
        else: wavs.append(torch.zeros(1, VAL_SAMPLES))
    return wavs

def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            for i in range(mel.size(0)): mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
            p_clip  = torch.sigmoid(clip_logits).cpu().numpy()
            p_fmax  = torch.sigmoid(frame_max).cpu().numpy()
            preds_clip.append(p_clip); preds_fmax.append(p_fmax)
            preds_blend.append(0.5 * p_clip + 0.5 * p_fmax)
    return {'clip':  np.concatenate(preds_clip),
            'fmax':  np.concatenate(preds_fmax),
            'blend': np.concatenate(preds_blend)}

def build_active_datasets(fold_k):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta['fold'] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(('focal', fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta['fold'].values == fold_k
        sds = ScDS(Y_SC[~vm], sc_cache_meta[~vm].reset_index(drop=True), aug=True)
        items.append(('sc', sds, len(sds)))
    return items

LOG_INTERVAL = 50

def train_fold(fold_k):
    vm       = sc_cache_meta['fold'].values == fold_k
    Y_val    = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    val_sc_df = sc_cache_meta[vm].reset_index(drop=True)

    # --- Perch プリ計算 (fold ごとに 1 回) ---
    print(f'[fold {fold_k}] Loading Perch teacher ...')
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH)
    print(f'[fold {fold_k}] Pre-computing Perch embeddings ...')
    precompute_all_embeddings(perch_teacher)
    del perch_teacher; gc.collect()
    print(f'[fold {fold_k}] Perch teacher released. Training is now GPU-bound.')

    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))
    print(f'  Streams: {dict(zip(names, sizes))}  steps/ep: {nst}')
    print(f'  Total: {EPOCHS} epochs x {nst} steps = {EPOCHS*nst} steps')

    m = make_model()
    mel_transform = MelSpecTransform().to(device)
    spec_augment  = SpecAugment().to(device)

    opt    = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = nst * WARMUP_EPOCHS
    total_steps  = nst * EPOCHS
    sch = torch.optim.lr_scheduler.SequentialLR(
        opt,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=1/25, end_factor=1.0, total_iters=warmup_steps),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup_steps, eta_min=1e-6),
        ],
        milestones=[warmup_steps])

    history = {'ep': [], 'train_loss': [], 'cls_loss': [], 'dist_loss': [],
               'ns22_macro': [], 'ns22_Aves': [], 'ns22_Amphibia': [],
               'ns22_Insecta': [], 'ns22_Mammalia': [], 'val_preds': []}
    best_ns22, best_state_ns22 = -1.0, None
    best_macro, best_state_macro = -1.0, None
    val_wavs = _load_val_waveforms(val_sc_df)

    for ep in range(EPOCHS):
        m.train()
        smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        tl  = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m,
                         num_workers=4, pin_memory=True)  # Colab: num_workers=4 で並列 I/O
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        t0 = time.time()
        print(f'  Ep{ep:02d} start [{time.strftime("%H:%M:%S")}]')

        for step, (wav, lb, wt, mk, sr, perch_emb) in enumerate(tl):
            wav, lb, wt, mk = wav.to(device), lb.to(device), wt.to(device), mk.to(device)
            perch_emb = perch_emb.to(device)  # プリ計算済み埋め込み
            sw = mk_sw(sr).to(device)

            with torch.no_grad():
                mel = mel_transform(wav)
                for i in range(mel.size(0)): mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)

            with autocast():
                if USE_PERCH_DISTILL:
                    clip_logits, framewise, distill_emb = m(mel, return_framewise=True, return_distill=True)
                else:
                    clip_logits, framewise = m(mel, return_framewise=True)

                frame_max_logits = framewise.max(dim=1).values
                bce = 0.5 * F.binary_cross_entropy_with_logits(clip_logits, lb, reduction='none') \
                    + 0.5 * F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction='none')
                cls_loss = ((bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8) * sw).mean()

                if USE_PERCH_DISTILL:
                    distill_loss = F.mse_loss(distill_emb, perch_emb)  # ONNX 推論なし！
                    loss = cls_loss + ALPHA_DISTILL * distill_loss
                else:
                    distill_loss = torch.tensor(0.0)
                    loss = cls_loss

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()

            el += loss.item(); el_cls += cls_loss.item(); el_dist += distill_loss.item(); nb_count += 1

            if (step + 1) % LOG_INTERVAL == 0:
                elapsed = time.time() - t0
                sps = (step + 1) / elapsed
                eta = (nst - step - 1) / sps
                dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
                print(f"  Ep{ep:02d} [{step+1:4d}/{nst}] "
                      f"loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
                      f"lr={opt.param_groups[0]['lr']:.1e} "
                      f"| {sps:.2f}it/s ETA:{eta/60:.1f}min [{time.strftime('%H:%M:%S')}]")

        # Validation
        val_preds_dict = _predict_from_waveforms(m, mel_transform, val_wavs)
        val_preds = val_preds_dict['blend']
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        for mode in ['clip', 'fmax', 'blend']:
            r[f'ns22_{mode}'] = full_eval(Y_val, val_preds_dict[mode], ns22_val, TAXON_MASKS)['non_s22_macro']

        history['ep'].append(ep)
        history['train_loss'].append(round(el / nb_count, 5))
        history['cls_loss'].append(round(el_cls / nb_count, 5))
        history['dist_loss'].append(round(el_dist / nb_count, 5))
        history['ns22_macro'].append(r['non_s22_macro'])
        for t in ['Aves', 'Amphibia', 'Insecta', 'Mammalia']:
            history[f'ns22_{t}'].append(r[f'non_s22_{t}'])
        history['val_preds'].append(val_preds.astype(np.float32))

        tag_ns22 = tag_macro = ''
        if r['non_s22_macro'] > best_ns22:
            best_ns22 = r['non_s22_macro']
            best_state_ns22 = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_ns22 = ' *ns22'
        if r['macro_auc_all'] > best_macro:
            best_macro = r['macro_auc_all']
            best_state_macro = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_macro = ' *macro'

        dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
        ep_time  = time.time() - t0
        print(f"    Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
              f"lr={opt.param_groups[0]['lr']:.1e} | "
              f"ns22: {r['ns22_blend']:.4f} | "
              f"Av={r['non_s22_Aves']:.4f} Am={r['non_s22_Amphibia']:.4f} "
              f"In={r['non_s22_Insecta']:.4f} Ma={r['non_s22_Mammalia']:.4f} "
              f"[{ep_time/60:.1f}min]{tag_ns22}{tag_macro}")

    del m, mel_transform, spec_augment
    torch.cuda.empty_cache(); gc.collect()
    return best_state_ns22, best_state_macro, history

print('OK Training function ready')

## S6 — Fold Loop + ONNX Export

In [ ]:
# =================================================================
# S6 -- FOLD LOOP + ONNX EXPORT
# =================================================================

if MODE != 'train':
    print("Skipping training (MODE='infer')")
    oof_ns22 = None; all_hist = {}
else:
    oof_ns22 = np.full((len(sc_cache_meta), NUM_CLASSES), np.nan, dtype=np.float32)
    all_hist = {}

    for fold_k in FOLDS:
        print(f"\n{'='*60}\nFOLD {fold_k}\n{'='*60}")
        vm = sc_cache_meta['fold'].values == fold_k
        val_sc_df_k = sc_cache_meta[vm].reset_index(drop=True)
        print(f'[S6] val windows: {vm.sum()}')

        best_ns22_state, best_macro_state, hist = train_fold(fold_k)
        print(f'[S6] train_fold done. best_macro={"set" if best_macro_state else "None"}')
        all_hist[fold_k] = hist

        mel_tf = MelSpecTransform().to(device)
        val_wavs_k = _load_val_waveforms(val_sc_df_k)

        if best_macro_state is not None:
            ckpt_path = OUT_DIR / f'fold{fold_k}_best_macro.pt'
            print(f'[S6] Saving checkpoint -> {ckpt_path}')
            torch.save(best_macro_state, ckpt_path)

            print('[S6] OOF prediction ...')
            m = make_model()
            m.load_state_dict(best_macro_state, strict=False)
            oof_ns22[vm] = _predict_from_waveforms(m, mel_tf, val_wavs_k)['blend']
            print(f'[S6] OOF done: shape={oof_ns22[vm].shape}')

            # ONNX export
            m.eval()
            INF_N_MELS   = 128
            INF_N_FRAMES = VAL_SAMPLES // HOP_LENGTH + 1

            class SEDExportWrapper(nn.Module):
                def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
                    super().__init__()
                    self.backbone = timm.create_model(backbone_name, pretrained=False, in_chans=1,
                                                      num_classes=0, global_pool='', drop_path_rate=0.1)
                    self.gem_freq    = GeMFreqPool(p_init=3.0)
                    self.dense_drop1 = nn.Dropout(0.25)
                    self.dense_conv  = nn.Conv1d(backbone_dim, hidden_dim, 1)
                    self.dense_relu  = nn.ReLU(inplace=True)
                    self.dense_drop2 = nn.Dropout(0.5)
                    self.att         = nn.Conv1d(hidden_dim, num_classes, 1)
                    self.cla         = nn.Conv1d(hidden_dim, num_classes, 1)
                def forward(self, mel):
                    h = self.backbone(mel)
                    h = self.gem_freq(h)
                    h = self.dense_drop1(h)
                    h = self.dense_conv(h)
                    h = self.dense_relu(h)
                    h = self.dense_drop2(h)
                    norm_att  = torch.softmax(torch.tanh(self.att(h)), dim=-1)
                    framewise = self.cla(h)
                    clip      = torch.sum(norm_att * framewise, dim=2)
                    return clip, framewise.permute(0, 2, 1)

            def load_and_remap_state(export_model, trained_state):
                remap = {}
                for k, v in trained_state.items():
                    if k.startswith('distill_head.'): continue
                    if k == 'dense.1.weight': remap['dense_conv.weight'] = v.unsqueeze(-1)
                    elif k == 'dense.1.bias': remap['dense_conv.bias'] = v
                    else: remap[k] = v
                export_model.load_state_dict(remap, strict=False)

            print(f'[S6] Building export model (backbone_dim={m.backbone_dim}) ...')
            export_model = SEDExportWrapper(BACKBONE_NAME, NUM_CLASSES, m.backbone_dim).to(device)
            load_and_remap_state(export_model, best_macro_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f'sed_distill_fold{fold_k}.onnx'
            print(f'[S6] Exporting ONNX -> {onnx_path} ...')
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=['mel'], output_names=['clip_logits', 'framewise_logits'],
                dynamic_axes={'mel': {0: 'batch'}, 'clip_logits': {0: 'batch'},
                              'framewise_logits': {0: 'batch'}},
                opset_version=17)

            print('[S6] Verifying ONNX ...')
            _sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
            _out  = _sess.run(None, {'mel': dummy_mel.cpu().numpy()})
            with torch.no_grad(): _ref, _ = export_model(dummy_mel)
            _diff = np.abs(_ref.cpu().numpy() - _out[0]).max()
            print(f'  ONNX verify: max|diff|={_diff:.3e}')
            assert _diff < 1e-3, f'ONNX diverged: {_diff}'
            del _sess

            size_mb = onnx_path.stat().st_size / 1e6
            print(f'  Exported {onnx_path.name} ({size_mb:.1f} MB)')
            del m, export_model
            print(f'[S6] Fold {fold_k} complete!')
        else:
            print(f'[S6] WARNING: best_macro_state is None for fold {fold_k}')

## S7 — OOF Evaluation

In [ ]:
# =================================================================
# S7 -- OOF EVALUATION
# =================================================================
if MODE == 'train' and oof_ns22 is not None:
    has = ~np.isnan(oof_ns22[:, 0])
    if has.sum() > 0:
        r_all = full_eval(Y_SC[has], oof_ns22[has], non_s22_mask_sc[has], TAXON_MASKS)
        print('=' * 60)
        print('OOF RESULTS')
        print('=' * 60)
        print(f"  macro AUC (all):      {r_all['macro_auc_all']:.4f}")
        print(f"  macro AUC (non-S22):  {r_all['non_s22_macro']:.4f}")
        for t in ['Aves', 'Amphibia', 'Insecta', 'Mammalia']:
            print(f"    {t:<12}: {r_all.get(f'non_s22_{t}', float('nan')):.4f}")

In [ ]:
# =================================================================
# S8 -- ONNX ダウンロード (Colab のみ)
# 学習完了後にこのセルを実行して ONNX ファイルをローカルに保存
# =================================================================
if IS_COLAB:
    from google.colab import files
    import glob
    onnx_files = sorted(glob.glob('/content/working/*.onnx'))
    if onnx_files:
        print('Downloading ONNX files:')
        for f in onnx_files:
            size_mb = os.path.getsize(f) / 1e6
            print(f'  {f}  ({size_mb:.1f} MB)')
            files.download(f)
    else:
        print('No ONNX files found in /content/working/')
        print('Available files:', os.listdir('/content/working/'))